In [41]:

#import the necessary libraries to execute this code
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# import model frameworks
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from ngboost import NGBRegressor
from sklearn.model_selection import RandomizedSearchCV as RSCV

class NESTED_CV:
  
    """
    NESTED_CV Class:
    - based on a dataset for NExT delivery systems
    - contains 10 different model architectures and non-exhaustive hyperparamater spaces for those models
    - once model type is selected, NESTED_CV will be conducted, data is split as follows:
          - outer_loop (test) done by GroupKFold where 20% of the dataset are held back at random
          - inner_loop (HP screening) done by GroupKFold based 10 splits in the dataset - based on feature groups
    - default is 10-folds for the NESTED_CV, but this can be entered manually
    - prints progress and reults at the end of each loop
    - configures a pandas dataframe with the reults of the NESTED_CV
    - fits and trains the best model based on the reults of the NESTED_CV
    """

    def __init__(self, datafile = "34_feat_dataset.xlsx", model_type = None):
        self.df = pd.read_excel(datafile)
          
        if model_type == 'MLR':
          self.user_defined_model = LinearRegression()
          self.p_grid = {'fit_intercept':[True, False],
                         'positive':[False]}
    
        elif model_type == 'lasso':
          self.user_defined_model = linear_model.Lasso()
          Pipeline([ ("scaler", StandardScaler()),("lasso", Lasso(max_iter=10000))])
          self.p_grid = {'alpha':[0.01, 0.02, 0.05, 0.1, 0.25, 0.5, 1.0, 10],
                        'positive':[True, False]}
            
        elif model_type == 'kNN':
          self.user_defined_model = KNeighborsRegressor()
          self.p_grid ={'n_neighbors':[2, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30, 50],
                        'weights': ["uniform", 'distance'],
                        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                        'leaf_size': [10, 30, 50, 75, 100],
                        'p':[1, 2],
                        'metric': ['minkowski']}

        elif model_type == 'PLS':
          self.user_defined_model = PLSRegression()
          self.p_grid ={'n_components':[2, 4, 6],
                        'max_iter': [250, 500, 750, 1000]}

        elif model_type == 'SVR':
          self.user_defined_model = SVR()
          self.p_grid ={'kernel':['linear', 'poly', 'rbf', 'sigmoid'],
                        'degree':[2, 3, 4, 5, 6],
                        'gamma':['scale', 'auto'],
                        'C':[0.1, 0.5, 1, 2],
                        'epsilon':[0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2],
                        'shrinking': [True, False]}

        elif model_type == 'DT':
          self.user_defined_model = DecisionTreeRegressor(random_state=4)
          self.p_grid ={'criterion':['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
                        'splitter':['best', 'random'],
                        'max_depth':[None],
                        'min_samples_split':[2,4,6],
                        'min_samples_leaf':[1,2,4],
                        'max_features': [None, 'sqrt','log2'],
                        'ccp_alpha': [0, 0.05, 0.1, 0.15]}  
        
        elif model_type == 'RF':
          self.user_defined_model = RandomForestRegressor(random_state=4)
          self.p_grid ={'n_estimators':[100,300,400],
                        'criterion':['squared_error', 'absolute_error'],
                        'max_depth':[None],
                        'min_samples_split':[2,4,6,8],
                        'min_samples_leaf':[1,2,4],
                        'min_weight_fraction_leaf':[0.0],
                        'max_features': ['sqrt', 'log2'],
                        'max_leaf_nodes':[None],
                        'min_impurity_decrease': [0.0],
                        'bootstrap':[True],
                        'oob_score':[True],
                        'ccp_alpha': [0, 0.005, 0.01]}

        elif model_type == 'LGBM':
          self.user_defined_model = LGBMRegressor(random_state=4,
                                                  #remove Row-wise multi-threading message 
                                                  force_row_wise=True,
                                                  verbose=-1)
          self.p_grid ={"n_estimators":[100,150,200,250,300,400,500,600],
                        'boosting_type': ['gbdt', 'dart', 'goss'],
                        'num_leaves':[16,32,64,128,256],
                        'learning_rate':[0.01,0.001,0.0001],
                        'min_child_weight': [0.001,0.01,0.1,1.0,10.0],
                        'subsample': [0.4,0.6,0.8,1.0],
                        'min_child_samples':[2,10,20,40,100],
                        'reg_alpha': [0, 0.005, 0.01, 0.015],
                        'reg_lambda': [0, 0.005, 0.01, 0.015]}
        
        elif model_type == 'XGB':
          self.user_defined_model = XGBRegressor(objective ='reg:squarederror', n_jobs=-1)
          self.p_grid ={'booster': ['gbtree', 'dart'],
                        "n_estimators":[100, 150, 300, 400],
                        'max_depth':[3, 4, 5, 6, 7, 8, 9, 10],
                        'gamma':[0, 2, 4, 6, 8, 10],
                        'learning_rate':[0.3, 0.2, 0.1, 0.05, 0.01],
                        'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                        'min_child_weight': [1.0, 2.0, 4.0, 5.0],
                        'max_delta_step':[1, 2, 4, 6, 8, 10],
                        'reg_alpha':[0.001, 0.01, 0.1],
                        'reg_lambda': [0.001, 0.01, 0.1]}                
        
        elif model_type == 'NGB':
          b1 = DecisionTreeRegressor(criterion='squared_error', max_depth=2)
          b2 = DecisionTreeRegressor(criterion='squared_error', max_depth=4)
          b3 = DecisionTreeRegressor(criterion='squared_error', max_depth=8) 
          b4 = DecisionTreeRegressor(criterion='squared_error', max_depth=12)
          b5 = DecisionTreeRegressor(criterion='squared_error', max_depth=16)
          b6 = DecisionTreeRegressor(criterion='squared_error', max_depth=32) 
          self.user_defined_model = NGBRegressor()
          self.p_grid ={'n_estimators':[100,200,300,400,500,600,800],
                        'learning_rate': [0.1, 0.01, 0.001],
                        'minibatch_frac': [1.0, 0.8, 0.5],
                        'col_sample': [1, 0.8, 0.5],
                        'Base': [b1, b2, b3, b4, b5, b6]}
        
        else:
          print("#######################\nSELECTION UNAVAILABLE!\n#######################\n\nPlease chose one of the following options:\n\n 'MLR'for multiple linear regression\n\n 'lasso' for multiple linear regression with east absolute shrinkage and selection operator (lasso)\n\n 'kNN'for k-Nearest Neighbors\n\n 'PLS' for partial least squares\n\n 'SVR' for support vertor regressor\n\n 'DT' for decision tree\n\n 'RF' for random forest\n\n 'LGBM' for LightGBM\n\n 'XGB' for XGBoost\n\n 'NGB' for NGBoost")

    def input_target(self):
        
        X = self.df.iloc[:, :-5]        
        stdScale = StandardScaler().fit(X)
        # for X to be a .pdframe instead of numpy array (LGBM)
        self.X = pd.DataFrame(
            stdScale.transform(X),
            columns=X.columns,
            index=X.index)
        
        outputs = {
            "Transfection efficiency": self.df.iloc[:, -3],
            "Cell viability": self.df.iloc[:, -2],
            "Mean Fluorescence Intensity": self.df.iloc[:, -1]
        }
        self.outputs = outputs
        
    def cross_validation(self, input_value):
        self.itr_number = []
        self.outer_results = []
        self.inner_results = []
        self.model_params = []
        self.pred_list = []
        self.mae_results = []
        self.rmse_results = []
        self.r2_results = []
        self.y_test_list = []
        self.output_name_list = []

        for output_name, Y in self.outputs.items():
            
            print("\nRunning model for:", output_name)
            self.Y = Y
            
            if input_value == None:
                NUM_TRIALS = 10
            else: 
                NUM_TRIALS = input_value



            for i in range(NUM_TRIALS): #configure the cross-validation procedure - outer loop (test set) 
  
              cv_outer = KFold(n_splits=5, shuffle=True, random_state=i) #hold back 20% of the groups for test set

              # split data using GSS
              for train_index, test_index in cv_outer.split(self.X):
                  #had to change self.X.iloc[train_index] due to self.X being a dataframe and not numpy Array for LGBM
                  #X_train, X_test = self.X[train_index], self.X[test_index]
                  #y_train, y_test = self.Y[train_index], self.Y[test_index]
                  X_train, X_test = self.X.iloc[train_index], self.X.iloc[test_index]
                  y_train, y_test = self.Y.iloc[train_index], self.Y.iloc[test_index]
                  
                  
                  # store test set information
                  y_test = np.array(y_test) #prevents index from being brought from dataframe
                  self.y_test_list.append(y_test)
                    
                  # configure the cross-validation procedure - inner loop (validation set/HP optimization)
                  cv_inner = KFold(n_splits=5, shuffle=True, random_state=i) #chosen 5 fold group split for inner loop

                  # define search space
                  search = RSCV(self.user_defined_model, self.p_grid, n_iter=100, verbose=0, scoring='neg_mean_absolute_error', cv=cv_inner,  n_jobs= 1, refit=True) # should be 100
                      
                  # execute search
                  result = search.fit(X_train, y_train)
                  
                  # get the best performing model fit on the whole training set
                  best_model = result.best_estimator_

                  # get the score for the best performing model and store
                  best_score = abs(result.best_score_)
                  self.inner_results.append(best_score)

                  # evaluate model on the hold out dataset
                  yhat = best_model.predict(X_test)
                  mae = mean_absolute_error(y_test, yhat)
                  rmse = np.sqrt(mean_squared_error(y_test, yhat))
                  r2 = r2_score(y_test, yhat)

                  # store output predictions
                  self.pred_list.append(yhat)
                  self.mae_results.append(mae)
                  self.rmse_results.append(rmse)
                  self.r2_results.append(r2)

              # evaluate the model
                  acc = mean_absolute_error(y_test, yhat)
                      
              # store the result
                  self.itr_number.append(i+1)
                  self.outer_results.append(acc)
                  self.output_name_list.append(output_name)
                  self.model_params.append(result.best_params_)

              # report progress at end of each inner loop
                  print('\n################################################################\n\nSTATUS REPORT:') 
                  print('Iteration '+str(i+1)+' of '+str(NUM_TRIALS)+' runs completed') 
                  print('Test_Score: %.3f, Best_Valid_Score: %.3f, \n\nBest_Model_Params: \n%s' % (acc, best_score, result.best_params_))
                  print("\n################################################################\n ")
          
    def results(self):   
        #create dataframe with results of nested CV
        
        list_of_tuples = list(zip(
        self.output_name_list,
        self.itr_number,
        self.inner_results,
        self.outer_results,
        self.model_params,
        self.y_test_list,
        self.pred_list,
        self.mae_results,
        self.rmse_results,
        self.r2_results))
   
        CV_dataset = pd.DataFrame(list_of_tuples, columns = ['Output', 'Iter', 'Valid Score', 'Test Score', 'Model Params', 'Experimental_Value', 'Predicted_Value', 'MAE', 'RMSE', 'R2'])
        CV_dataset['Score_difference'] = abs(CV_dataset['Valid Score'] - CV_dataset['Test Score']) #Groupby dataframe model iterations that best fit the data (i.e., validitaion <= test)
        
        CV_dataset.sort_values(by=['Score_difference', 'Test Score'], ascending=True, inplace=True) 
        CV_dataset = CV_dataset.reset_index(drop=True) # Reset index of dataframe
        # CV_dataset.groupby("Output")["Test Score"].mean()
        
        # save the results as a class object
        self.CV_dataset = CV_dataset

    def check_best_params(self):
        print("\n Checking best hyperparameters per output:\n")
    
        for output_name in self.CV_dataset["Output"].unique():
            df_output = self.CV_dataset[self.CV_dataset["Output"] == output_name]
            best_params = df_output.iloc[0, 4]
        
            print(f"{output_name}:")
            print(best_params)
            print("-" * 50)
    
    def check_predictions(self):
        print("\n Checking predictions per output:\n")
    
        for name, model in self.best_models.items():
            preds = model.predict(self.X)
            print(name, np.round(preds[:5], 4))
            
    def best_model(self):
        import copy
        self.best_models = {}
        
        for output_name in self.CV_dataset["Output"].unique():
            df_output = self.CV_dataset[self.CV_dataset["Output"] == output_name]
            best_model_params = df_output.iloc[0, 4]

            #create an independent model
            model = copy.deepcopy(self.user_defined_model)

            # set params on the new instance
            model.set_params(**best_model_params)

            Y = self.outputs[output_name]
            trained_model = model.fit(self.X, Y)
            self.best_models[output_name] = trained_model


In [30]:
import pickle
from sklearn.model_selection import RandomizedSearchCV

def run_NESTED_CV(name, CV):

  if __name__ == '__main__':
    model_instance = NESTED_CV(model_type = name)
    model_instance.input_target()
    #model_instance.cross_validation(CV)
    #model_instance.results()
    model_instance.CV_dataset = pd.read_pickle("nested_CV_pkl/NESTED_CV_RESULTS_34_feat_"+str(name)+".pkl")
    model_instance.best_model()
    model_instance.check_best_params()
    model_instance.check_predictions()
    
    #model_instance.CV_dataset.to_pickle("NESTED_CV_RESULTS_34_feat_"+str(name)+".pkl")
    
    with open('trained_models2_pkl/trained_models2_34_feat_'+str(name)+'_fly.pkl', 'wb') as file: # Save the Model to pickle file
        pickle.dump(model_instance.best_models, file)

    return model_instance.CV_dataset, model_instance.best_models


In [31]:
NESTED_CV()

#######################
SELECTION UNAVAILABLE!
#######################

Please chose one of the following options:

 'MLR'for multiple linear regression

 'lasso' for multiple linear regression with east absolute shrinkage and selection operator (lasso)

 'kNN'for k-Nearest Neighbors

 'PLS' for partial least squares

 'SVR' for support vertor regressor

 'DT' for decision tree

 'RF' for random forest

 'LGBM' for LightGBM

 'XGB' for XGBoost

 'NGB' for NGBoost


In [32]:
XGB_result, XGB_model = run_NESTED_CV("XGB", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'subsample': 0.8, 'reg_lambda': 0.001, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_weight': 2.0, 'max_depth': 8, 'max_delta_step': 2, 'learning_rate': 0.01, 'gamma': 0, 'booster': 'dart'}
--------------------------------------------------
Cell viability:
{'subsample': 1.0, 'reg_lambda': 0.001, 'reg_alpha': 0.01, 'n_estimators': 300, 'min_child_weight': 1.0, 'max_depth': 4, 'max_delta_step': 6, 'learning_rate': 0.05, 'gamma': 4, 'booster': 'gbtree'}
--------------------------------------------------
Transfection efficiency:
{'subsample': 1.0, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'n_estimators': 400, 'min_child_weight': 1.0, 'max_depth': 4, 'max_delta_step': 10, 'learning_rate': 0.05, 'gamma': 2, 'booster': 'dart'}
--------------------------------------------------

 Checking predictions per output:

Mean Fluorescence Intensity [5.6739 6.6588 6.2205 5.645  6.6103]
Cell viability [95.4388 95.7051 95.9642 94.6

In [33]:
LGBM_result, LGBM_model = run_NESTED_CV("LGBM", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'subsample': 0.8, 'reg_lambda': 0.005, 'reg_alpha': 0.01, 'num_leaves': 64, 'n_estimators': 500, 'min_child_weight': 0.001, 'min_child_samples': 2, 'learning_rate': 0.01, 'boosting_type': 'gbdt'}
--------------------------------------------------
Cell viability:
{'subsample': 0.4, 'reg_lambda': 0.005, 'reg_alpha': 0.015, 'num_leaves': 16, 'n_estimators': 600, 'min_child_weight': 0.1, 'min_child_samples': 2, 'learning_rate': 0.01, 'boosting_type': 'gbdt'}
--------------------------------------------------
Transfection efficiency:
{'subsample': 0.6, 'reg_lambda': 0.015, 'reg_alpha': 0.005, 'num_leaves': 32, 'n_estimators': 400, 'min_child_weight': 10.0, 'min_child_samples': 10, 'learning_rate': 0.01, 'boosting_type': 'gbdt'}
--------------------------------------------------

 Checking predictions per output:

Mean Fluorescence Intensity [5.5771 6.8184 6.3139 5.3383 6.7255]
Cell viability [94.8723 95.3751 95.261  

In [34]:
RF_result, RF_model = run_NESTED_CV("RF", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'oob_score': True, 'n_estimators': 400, 'min_weight_fraction_leaf': 0.0, 'min_samples_split': 6, 'min_samples_leaf': 1, 'min_impurity_decrease': 0.0, 'max_leaf_nodes': None, 'max_features': 'log2', 'max_depth': None, 'criterion': 'absolute_error', 'ccp_alpha': 0, 'bootstrap': True}
--------------------------------------------------
Transfection efficiency:
{'oob_score': True, 'n_estimators': 400, 'min_weight_fraction_leaf': 0.0, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_impurity_decrease': 0.0, 'max_leaf_nodes': None, 'max_features': 'log2', 'max_depth': None, 'criterion': 'squared_error', 'ccp_alpha': 0, 'bootstrap': True}
--------------------------------------------------
Cell viability:
{'oob_score': True, 'n_estimators': 400, 'min_weight_fraction_leaf': 0.0, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_impurity_decrease': 0.0, 'max_leaf_nodes': None, 'max_features': 'sqrt', 'max_depth': None

In [35]:
DT_result, DT_model = run_NESTED_CV("DT", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'splitter': 'random', 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': None, 'criterion': 'absolute_error', 'ccp_alpha': 0}
--------------------------------------------------
Cell viability:
{'splitter': 'random', 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': None, 'criterion': 'friedman_mse', 'ccp_alpha': 0.1}
--------------------------------------------------
Transfection efficiency:
{'splitter': 'random', 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None, 'criterion': 'absolute_error', 'ccp_alpha': 0.05}
--------------------------------------------------

 Checking predictions per output:

Mean Fluorescence Intensity [5.0769 5.0769 7.4225 6.7124 6.7124]
Cell viability [98.1171 98.1171 98.1171 95.7236 93.6165]
Transfection efficiency [29.225 29.225 29.225 29.225 29.225]


In [36]:
SVR_result, SVR_model = run_NESTED_CV("SVR", None) 


 Checking best hyperparameters per output:

Cell viability:
{'shrinking': True, 'kernel': 'rbf', 'gamma': 'scale', 'epsilon': 0.001, 'degree': 2, 'C': 2}
--------------------------------------------------
Transfection efficiency:
{'shrinking': True, 'kernel': 'linear', 'gamma': 'scale', 'epsilon': 0.005, 'degree': 6, 'C': 1}
--------------------------------------------------
Mean Fluorescence Intensity:
{'shrinking': False, 'kernel': 'rbf', 'gamma': 'auto', 'epsilon': 0.05, 'degree': 5, 'C': 2}
--------------------------------------------------

 Checking predictions per output:

Cell viability [96.6387 96.1694 97.1002 94.4511 95.3988]
Transfection efficiency [26.4647 24.0408 27.7353 25.6336 34.3055]
Mean Fluorescence Intensity [5.6529 6.7909 6.4383 5.2949 6.7459]


In [37]:
PLS_result, PLS_model = run_NESTED_CV("PLS", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'n_components': 2, 'max_iter': 250}
--------------------------------------------------
Cell viability:
{'n_components': 6, 'max_iter': 250}
--------------------------------------------------
Transfection efficiency:
{'n_components': 6, 'max_iter': 250}
--------------------------------------------------

 Checking predictions per output:

Mean Fluorescence Intensity [3.2062 4.6961 6.2708 5.8387 7.5755]
Cell viability [92.9499 94.1353 91.3645 92.7667 92.6362]
Transfection efficiency [23.2093 20.3676 24.7681 24.7343 32.1305]


In [38]:
lasso_result, lasso_model = run_NESTED_CV("lasso", None) 


 Checking best hyperparameters per output:

Cell viability:
{'positive': False, 'alpha': 0.01}
--------------------------------------------------
Mean Fluorescence Intensity:
{'positive': False, 'alpha': 0.02}
--------------------------------------------------
Transfection efficiency:
{'positive': False, 'alpha': 0.25}
--------------------------------------------------

 Checking predictions per output:

Cell viability [94.1223 95.3864 92.0376 92.3991 93.4358]
Mean Fluorescence Intensity [4.3713 5.4037 7.0383 5.6519 7.1817]
Transfection efficiency [21.0602 21.3296 23.3209 24.0872 31.0532]


In [39]:
MLR_result, MLR_model = run_NESTED_CV("MLR", None) 


 Checking best hyperparameters per output:

Cell viability:
{'positive': False, 'fit_intercept': True}
--------------------------------------------------
Mean Fluorescence Intensity:
{'positive': False, 'fit_intercept': True}
--------------------------------------------------
Transfection efficiency:
{'positive': False, 'fit_intercept': True}
--------------------------------------------------

 Checking predictions per output:

Cell viability [94.3142 95.1026 93.6758 91.6798 92.8046]
Mean Fluorescence Intensity [4.9286 5.4061 7.6407 5.5597 6.9465]
Transfection efficiency [24.6275 19.9537 28.2184 25.069  31.9504]


In [40]:
kNN_result, kNN_model = run_NESTED_CV("kNN", None) 


 Checking best hyperparameters per output:

Mean Fluorescence Intensity:
{'weights': 'distance', 'p': 1, 'n_neighbors': 2, 'metric': 'minkowski', 'leaf_size': 75, 'algorithm': 'kd_tree'}
--------------------------------------------------
Cell viability:
{'weights': 'distance', 'p': 1, 'n_neighbors': 4, 'metric': 'minkowski', 'leaf_size': 100, 'algorithm': 'ball_tree'}
--------------------------------------------------
Transfection efficiency:
{'weights': 'distance', 'p': 1, 'n_neighbors': 2, 'metric': 'minkowski', 'leaf_size': 50, 'algorithm': 'auto'}
--------------------------------------------------

 Checking predictions per output:

Mean Fluorescence Intensity [5.6027 6.8412 6.3882 5.2452 6.7955]
Cell viability [96.64 96.17 98.01 94.45 95.4 ]
Transfection efficiency [26.47 28.63 27.73 23.65 34.31]


In [ ]:
NGB_result, NGB_model = run_NESTED_CV("NGB", None) 